# Practice - 01 - Bronze to Silver: types, nulls, quarantine, and deduplication

Prerequisites / execution context: Run 00 first. Uses only the dedicated kit schemas. Re-running replaces Silver exercise outputs.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R19: PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics); [R23: dropDuplicates](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/dropDuplicates); [R24: row_number](https://docs.databricks.com/aws/en/pyspark/reference/functions/row_number); [R25: NULL semantics](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-null-semantics).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo_practice"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Coding task 1: Clean and quarantine

Read BRONZE.orders_raw as raw. Build clean with amount_decimal DECIMAL(12,2), event_ts TIMESTAMP, trimmed customer_id, and quality_reason. Classify malformed money, negative money, bad timestamps and missing order IDs in that order. Missing customer_id is allowed. Produce quarantine and valid; write quarantine to SILVER.orders_quarantine.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
raw = spark.table(f"{BRONZE}.orders_raw")

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert valid.count() + quarantine.count() == raw.count()
assert quarantine.count() == 2
assert valid.count() == 6
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDEuIE1ha2UgaW52YWxpZCBkYXRhIG9ic2VydmFibGUKVXNlIHRyeV9jYXN0IHRvIGlkZW50aWZ5IG1hbGZvcm1lZCBtb25leSB3aXRob3V0IGFib3J0aW5nIHRoZSBmdWxsIGJhdGNoLiBUcmVhdCBtaXNzaW5nIGN1c3RvbWVyIElEcyBhcyBhIHNlcGFyYXRlIGVucmljaG1lbnQgY29uY2VybjsgdGhlIG9yZGVyIGl0c2VsZiBjYW4gcmVtYWluIHZhbGlkLgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPnJhdyA9IHNwYXJrLnRhYmxlKGYmcXVvdDt7QlJPTlpFfS5vcmRlcnNfcmF3JnF1b3Q7KQpjbGVhbiA9IChyYXcud2l0aENvbHVtbigmcXVvdDthbW91bnRfZGVjaW1hbCZxdW90OywgRi5leHByKCZxdW90O3RyeV9jYXN0KGFtb3VudCBBUyBERUNJTUFMKDEyLDIpKSZxdW90OykpCiAud2l0aENvbHVtbigmcXVvdDtldmVudF90cyZxdW90OywgRi5leHByKCZxdW90O3RyeV9jYXN0KGV2ZW50X3RpbWUgQVMgVElNRVNUQU1QKSZxdW90OykpCiAud2l0aENvbHVtbigmcXVvdDtjdXN0b21lcl9pZCZxdW90OywgRi50cmltKCZxdW90O2N1c3RvbWVyX2lkJnF1b3Q7KSkKIC53aXRoQ29sdW1uKCZxdW90O3F1YWxpdHlfcmVhc29uJnF1b3Q7LCBGLndoZW4oRi5jb2woJnF1b3Q7YW1vdW50X2RlY2ltYWwmcXVvdDspLmlzTnVsbCgpLCAmcXVvdDtiYWRfYW1vdW50JnF1b3Q7KQogICAgICAgICAgICAgLndoZW4oRi5jb2woJnF1b3Q7YW1vdW50X2RlY2ltYWwmcXVvdDspICZsdDsgMCwgJnF1b3Q7bmVnYXRpdmVfYW1vdW50JnF1b3Q7KQogICAgICAgICAgICAgLndoZW4oRi5jb2woJnF1b3Q7ZXZlbnRfdHMmcXVvdDspLmlzTnVsbCgpLCAmcXVvdDtiYWRfdGltZXN0YW1wJnF1b3Q7KQogICAgICAgICAgICAgLndoZW4oRi5jb2woJnF1b3Q7b3JkZXJfaWQmcXVvdDspLmlzTnVsbCgpLCAmcXVvdDttaXNzaW5nX29yZGVyX2lkJnF1b3Q7KSkpCnF1YXJhbnRpbmUgPSBjbGVhbi5maWx0ZXIoJnF1b3Q7cXVhbGl0eV9yZWFzb24gSVMgTk9UIE5VTEwmcXVvdDspCnZhbGlkID0gY2xlYW4uZmlsdGVyKCZxdW90O3F1YWxpdHlfcmVhc29uIElTIE5VTEwmcXVvdDspCmFzc2VydCBxdWFyYW50aW5lLmNvdW50KCk9PTIKYXNzZXJ0IHZhbGlkLmNvdW50KCk9PTYKcXVhcmFudGluZS53cml0ZS5tb2RlKCZxdW90O292ZXJ3cml0ZSZxdW90Oykuc2F2ZUFzVGFibGUoZiZxdW90O3tTSUxWRVJ9Lm9yZGVyc19xdWFyYW50aW5lJnF1b3Q7KQo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Coding task 2: Keep the current order

From valid, remove exact duplicates; drop quality_reason, raw amount and raw event_time; rename amount_decimal to amount. Name this DataFrame events. Keep the latest row per order_id using descending seq, event_ts, event_id; call the result latest. Write SILVER.orders. Define a tie-breaking contract.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert events.count() == 5
assert latest.count() == 4
assert spark.table(f"{SILVER}.orders").count() == 4
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDIuIENob29zZSB0aGUgbGF0ZXN0IHJlY29yZCBkZXRlcm1pbmlzdGljYWxseQpSZW1vdmUgZXhhY3QgZHVwbGljYXRlIGV2ZW50cyBmaXJzdC4gVGhlbiBvcmRlciBieSBzZXF1ZW5jZSwgdGltZXN0YW1wLCBhbmQgZXZlbnQgSUQuIEZvciB0aGlzIGRhdGFzZXQgdGhlIGZpbmFsIHRpZSBicmVha2VyIGlzIHVuaXF1ZSBhZnRlciBleGFjdCBkZWR1cGxpY2F0aW9uLiBBIHByb2R1Y3Rpb24gY29udHJhY3QgbXVzdCBkZWZpbmUgYSB0b3RhbCBvcmRlcmluZyB3aGVuIGNvbmZsaWN0aW5nIHBheWxvYWRzIHNoYXJlIGFsbCBvcmRlcmluZyB2YWx1ZXMuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9weXNwYXJrL3JlZmVyZW5jZS9mdW5jdGlvbnMvcm93X251bWJlciIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIyNDogcm93X251bWJlcjwvYT4uCgoKVGhlIHJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbiBpcyBzaG93biBuZXh0LiBDb21wYXJlIHlvdXIgZGVzaWduIGFuZCByZXN1bHRzIGJlZm9yZSBjb3B5aW5nIGl0LjwvZGl2PjwvZGV0YWlscz4=')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmV2ZW50cyA9IHZhbGlkLmRyb3BEdXBsaWNhdGVzKCkuZHJvcCgmcXVvdDtxdWFsaXR5X3JlYXNvbiZxdW90OywgJnF1b3Q7YW1vdW50JnF1b3Q7LCAmcXVvdDtldmVudF90aW1lJnF1b3Q7KS53aXRoQ29sdW1uUmVuYW1lZCgmcXVvdDthbW91bnRfZGVjaW1hbCZxdW90OywmcXVvdDthbW91bnQmcXVvdDspCncgPSBXaW5kb3cucGFydGl0aW9uQnkoJnF1b3Q7b3JkZXJfaWQmcXVvdDspLm9yZGVyQnkoRi5kZXNjKCZxdW90O3NlcSZxdW90OyksRi5kZXNjKCZxdW90O2V2ZW50X3RzJnF1b3Q7KSxGLmRlc2MoJnF1b3Q7ZXZlbnRfaWQmcXVvdDspKQpsYXRlc3QgPSBldmVudHMud2l0aENvbHVtbigmcXVvdDtybiZxdW90OywgRi5yb3dfbnVtYmVyKCkub3Zlcih3KSkuZmlsdGVyKCZxdW90O3JuID0gMSZxdW90OykuZHJvcCgmcXVvdDtybiZxdW90OykKYXNzZXJ0IGV2ZW50cy5jb3VudCgpPT01CmFzc2VydCBsYXRlc3QuY291bnQoKT09NAphc3NlcnQgbGF0ZXN0LmZpbHRlcigmcXVvdDtvcmRlcl9pZD0mI3gyNztvMSYjeDI3OyZxdW90OykuZmlyc3QoKS5hbW91bnQgPT0gX19pbXBvcnRfXygmI3gyNztkZWNpbWFsJiN4Mjc7KS5EZWNpbWFsKCYjeDI3OzEyMC4wMCYjeDI3OykKbGF0ZXN0LndyaXRlLmZvcm1hdCgmcXVvdDtkZWx0YSZxdW90OykubW9kZSgmcXVvdDtvdmVyd3JpdGUmcXVvdDspLnNhdmVBc1RhYmxlKGYmcXVvdDt7U0lMVkVSfS5vcmRlcnMmcXVvdDspCmxhdGVzdC5vcmRlckJ5KCZxdW90O29yZGVyX2lkJnF1b3Q7KS5zaG93KHRydW5jYXRlPUZhbHNlKQo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Coding task 3: Summarize and compare nulls

Use latest to compute row count, non-null customer count, approximate distinct customer count and mean amount. Show amount summary statistics. Demonstrate ordinary NULL equality versus null-safe equality.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDMuIE51bGxzIGFuZCBiYXNpYyBzdW1tYXJpZXMKQ291bnQgcm93cyBzZXBhcmF0ZWx5IGZyb20gbm9uLW51bGwgY3VzdG9tZXIgSURzLiBVc2UgYSBudWxsLXNhZmUgY29tcGFyaXNvbiBmb3IgbWlncmF0aW9uIG1pc21hdGNoIGxvZ2ljLiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vc3FsL2xhbmd1YWdlLW1hbnVhbC9zcWwtcmVmLW51bGwtc2VtYW50aWNzIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjI1OiBOVUxMIHNlbWFudGljczwvYT47IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vcHlzcGFyay9yZWZlcmVuY2UvY2xhc3Nlcy9kYXRhZnJhbWUiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNzQ6IERhdGFGcmFtZSByZWZlcmVuY2U8L2E+LgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmxhdGVzdC5zZWxlY3QoRi5jb3VudCgmcXVvdDsqJnF1b3Q7KS5hbGlhcygmcXVvdDtyb3dzJnF1b3Q7KSxGLmNvdW50KCZxdW90O2N1c3RvbWVyX2lkJnF1b3Q7KS5hbGlhcygmcXVvdDtrbm93bl9jdXN0b21lcl9pZHMmcXVvdDspLAogRi5hcHByb3hfY291bnRfZGlzdGluY3QoJnF1b3Q7Y3VzdG9tZXJfaWQmcXVvdDspLmFsaWFzKCZxdW90O2FwcHJveF9jdXN0b21lcl9pZHMmcXVvdDspLEYubWVhbigmcXVvdDthbW91bnQmcXVvdDspLmFsaWFzKCZxdW90O21lYW5fYW1vdW50JnF1b3Q7KSkuc2hvdygpCmxhdGVzdC5zZWxlY3QoJnF1b3Q7YW1vdW50JnF1b3Q7KS5zdW1tYXJ5KCZxdW90O2NvdW50JnF1b3Q7LCZxdW90O21pbiZxdW90OywmcXVvdDttYXgmcXVvdDssJnF1b3Q7bWVhbiZxdW90Oykuc2hvdygpCnNwYXJrLnNxbCgmcXVvdDtTRUxFQ1QgTlVMTCA9IE5VTEwgQVMgb3JkaW5hcnksIE5VTEwgJmx0Oz0mZ3Q7IE5VTEwgQVMgbnVsbF9zYWZlJnF1b3Q7KS5zaG93KCkKYXNzZXJ0IGxhdGVzdC5jb3VudCgpPT00IGFuZCBsYXRlc3QuZmlsdGVyKCZxdW90O2N1c3RvbWVyX2lkIElTIE5PVCBOVUxMJnF1b3Q7KS5jb3VudCgpPT0zCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Written / design task 1

Why is key-only dropDuplicates insufficient for latest-record logic? Why is replacing malformed money with zero dangerous? Reconcile raw, rejected, duplicate, superseded and current rows.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgQ2hhbGxlbmdlIGFuZCBzb2x1dGlvbgpXaHkgZG9lcyBkcm9wRHVwbGljYXRlcyhbJnF1b3Q7b3JkZXJfaWQmcXVvdDtdKSBub3QgZXhwcmVzcyB0aGUgbGF0ZXN0LXJlY29yZCByZXF1aXJlbWVudD8gSXQgZG9lcyBub3QgZW5jb2RlIHdoaWNoIHZlcnNpb24gdG8ga2VlcC4gV2h5IGlzIGZpbGxuYSgwKSBvbiBtb25leSBkYW5nZXJvdXMgaGVyZT8gSXQgd291bGQgaGlkZSBhIG1hbGZvcm1lZCBhbW91bnQgYXMgYSBnZW51aW5lIHplcm8uIFJlY29uY2lsaWF0aW9uOiA4IHJhdyA9IDIgcmVqZWN0ZWQgKyAxIGV4YWN0IGR1cGxpY2F0ZSByZW1vdmVkICsgMSBvbGRlciB2YWxpZCB2ZXJzaW9uIHJlbW92ZWQgKyA0IGN1cnJlbnQgb3JkZXJzLgo8L2Rpdj48L2RldGFpbHM+')
